In [5]:
# ============================================
# 0. 라이브러리 & DB 설정
# ============================================
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

SCHEMA_MACHINE = "d1_machine_log"

TABLES_FCT = [
    ("FCT1_machine_log", "FCT1"),
    ("FCT2_machine_log", "FCT2"),
    ("FCT3_machine_log", "FCT3"),
    ("FCT4_machine_log", "FCT4"),
]

NG_TEXT  = "제품 감지 NG"
OFF_TEXT = "제품 검사 투입요구 ON"

MANUAL_TEXT = "Manual mode 전환"
AUTO_TEXT   = "Auto mode 전환"


def get_engine(config=DB_CONFIG):
    user = config["user"]
    password = urllib.parse.quote_plus(config["password"])
    host = config["host"]
    port = config["port"]
    dbname = config["dbname"]
    conn_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(conn_str)

engine = get_engine()

# ============================================
# 1. FCT1~4 로그 로드 + station 컬럼 추가
# ============================================
def load_fct_log(table_name: str, station_label: str) -> pd.DataFrame:
    sql = f"""
        SELECT
            end_day,
            time,
            contents,
            dayornight
        FROM {SCHEMA_MACHINE}."{table_name}"
        ORDER BY time ASC
    """
    df = pd.read_sql(sql, engine)
    df["station"] = station_label
    return df

df_list = [load_fct_log(t, s) for t, s in TABLES_FCT]
df_all = pd.concat(df_list, ignore_index=True)

# ============================================
# 2. NG / OFF / Manual / Auto 이벤트만 추출 후 정렬
# ============================================
df_evt = df_all[
    df_all["contents"].isin([NG_TEXT, OFF_TEXT, MANUAL_TEXT, AUTO_TEXT])
].copy()

df_evt = df_evt.sort_values(
    ["end_day", "station", "time"]
).reset_index(drop=True)

# ============================================
# 3. 계산 로직
# ============================================
result_rows = []

for (end_day, station), grp in df_evt.groupby(["end_day", "station"], sort=False):
    pending_from_ts = None
    pending_from_dorn = None
    in_manual = False

    for _, row in grp.iterrows():
        contents = row["contents"]
        t = row["time"]
        dorn = row["dayornight"]

        ts = pd.to_datetime(f"{end_day} {t}")

        # Manual / Auto 상태 관리
        if contents == MANUAL_TEXT:
            in_manual = True
            continue
        if contents == AUTO_TEXT:
            in_manual = False
            continue

        # NG 처리
        if contents == NG_TEXT:
            if in_manual:
                continue

            # 연속 NG → 첫 NG만 유지
            if pending_from_ts is None:
                pending_from_ts = ts
                pending_from_dorn = dorn
            continue

        # OFF 처리
        if contents == OFF_TEXT and pending_from_ts is not None:
            from_ts = pending_from_ts
            to_ts = ts

            from_str = from_ts.strftime("%H:%M:%S.%f")[:-4]
            to_str   = to_ts.strftime("%H:%M:%S.%f")[:-4]

            wasted = round(abs((to_ts - from_ts).total_seconds()), 2)

            result_rows.append(
                {
                    "end_day": end_day,
                    "station": station,
                    "from_contents": NG_TEXT,
                    "from_time": from_str,
                    "from_dorn": pending_from_dorn,
                    "to_contents": OFF_TEXT,
                    "to_time": to_str,
                    "to_dorn": dorn,
                    "wasted_time": wasted,
                }
            )

            pending_from_ts = None
            pending_from_dorn = None

# ============================================
# 4. 결과 DataFrame 구성
# ============================================
df_wasted = pd.DataFrame(result_rows)

df_wasted = df_wasted.sort_values(
    ["end_day", "station", "from_time"]
).reset_index(drop=True)

df_wasted.insert(0, "id", range(1, len(df_wasted) + 1))

from IPython.display import display
display(df_wasted)

,id,end_day,station,from_contents,from_time,from_dorn,to_contents,to_time,to_dorn,wasted_time
0,1,20251001,FCT3,제품 감지 NG,19:26:54.98,day,제품 검사 투입요구 ON,19:27:16.76,day,21.78
1,2,20251030,FCT3,제품 감지 NG,21:41:38.61,night,제품 검사 투입요구 ON,21:41:53.63,night,15.02
2,3,20251030,FCT4,제품 감지 NG,12:03:06.05,day,제품 검사 투입요구 ON,12:04:43.66,day,97.61
3,4,20251030,FCT4,제품 감지 NG,12:04:58.66,day,제품 검사 투입요구 ON,12:05:55.44,day,56.78
4,5,20251030,FCT4,제품 감지 NG,12:06:18.98,day,제품 검사 투입요구 ON,12:08:12.06,day,113.08
5,6,20251030,FCT4,제품 감지 NG,15:21:37.65,day,제품 검사 투입요구 ON,15:21:49.61,day,11.96
6,7,20251030,FCT4,제품 감지 NG,15:26:24.83,day,제품 검사 투입요구 ON,15:26:40.58,day,15.75
7,8,20251030,FCT4,제품 감지 NG,15:27:09.24,day,제품 검사 투입요구 ON,15:27:17.58,day,8.34
8,9,20251030,FCT4,제품 감지 NG,15:27:31.01,day,제품 검사 투입요구 ON,15:27:39.85,day,8.84
9,10,20251031,FCT4,제품 감지 NG,23:04:06.48,night,제품 검사 투입요구 ON,23:04:19.39,night,12.91


In [6]:
# ============================================
# 5. 결과를 DB 테이블(d1_machine_log.afa_fail_wasted_time)에 저장
#    - 기존 테이블이 있으면 덮어씀(if_exists="replace")
#    - 기존 데이터에 추가하려면 "append" 로 변경
# ============================================
TABLE_SAVE_SCHEMA = "d1_machine_log"
TABLE_SAVE_NAME   = "afa_fail_wasted_time"

# 필요시 station, from_time, to_time 등의 dtype을 조정해도 됨
df_wasted.to_sql(
    TABLE_SAVE_NAME,
    con=engine,
    schema=TABLE_SAVE_SCHEMA,
    if_exists="replace",   # 기존 테이블을 새로 생성(덮어쓰기). 누적 저장이면 "append"
    index=False,
)

print(f"[DONE] {TABLE_SAVE_SCHEMA}.{TABLE_SAVE_NAME} 테이블에 {len(df_wasted)}행 저장 완료")

[DONE] d1_machine_log.afa_fail_wasted_time 테이블에 21행 저장 완료
